# Week 2 — Data Exploration

Explore historical real-estate sales data from CRMLS (California Regional Multiple Listing Service). The data includes final sale prices (`ClosePrice`) and property features such as living area, bedrooms, bathrooms, and lot size for California residential single-family homes.

- **Training / EDA:** January 2024–February 2026
- **Validation:** March 2026
- **Test:** April 2026

Only the training period is loaded for EDA. 


In [ ]:
# Setup
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
# Locate the project and data folders.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.paths import sold_data_dir

DATA_DIR = sold_data_dir()
print(DATA_DIR)

In [ ]:
# Discover and split the monthly files.
# Parse each filename's month.
def source_month(file):
    stamp = file.stem.replace("CRMLSSold", "").replace("_filled", "")
    return pd.Period(f"{stamp[:4]}-{stamp[4:6]}", freq="M")

csv_files = sorted(DATA_DIR.glob("CRMLSSold*.csv"), key=source_month)
file_months = [source_month(file) for file in csv_files]
print(f"found {len(csv_files)} monthly files")


In [ ]:
# Verify complete monthly coverage.
expected_months = list(pd.period_range("2024-01", "2026-04", freq="M"))
assert file_months == expected_months, "Expected one file per month from 2024-01 to 2026-04."

display(pd.DataFrame({
    "SourceMonth": file_months,
    "FileName": [file.name for file in csv_files],
}))


In [ ]:
# Create the chronological split.
# Reserve the final two months for validation and testing.
train_files = csv_files[:-2]
validation_file = csv_files[-2]
test_file = csv_files[-1]

display(pd.DataFrame({
    "Split": ["Training", "Validation", "Test"],
    "Months": [
        f"{source_month(train_files[0])} to {source_month(train_files[-1])}",
        str(source_month(validation_file)),
        str(source_month(test_file)),
    ],
    "Files": [len(train_files), 1, 1],
}))


In [ ]:
# Check all column names available across the training files.
if "train_files" not in globals():
    raise RuntimeError("Run the cells above first to create train_files.")

all_training_columns = sorted({
    column
    for file in train_files
    for column in pd.read_csv(file, nrows=0).columns
})
training_row_count = sum(
    len(chunk)
    for file in train_files
    for chunk in pd.read_csv(file, usecols=["ListingKey"], chunksize=100_000)
)

print(f"Full training source: {training_row_count:,} rows × {len(all_training_columns)} unique columns")
display(pd.DataFrame({"Column": all_training_columns}))


In [ ]:
# Preview the data and show each column's inferred type and an example value.
training_sample = pd.concat(
    [pd.read_csv(file, nrows=100, low_memory=False) for file in train_files],
    ignore_index=True,
    sort=False,
)
display(training_sample.head())

column_details = pd.DataFrame({
    "Inferred type": training_sample.dtypes.astype(str),
    "Example": [
        training_sample[column].dropna().iloc[0]
        if training_sample[column].notna().any() else pd.NA
        for column in training_sample.columns
    ],
})
column_details.index.name = "Column"
display(column_details)


In [ ]:
# Load the training data.
# ListPrice is checked for leakage but is not used as a model feature.
EDA_COLUMNS = [
    "ClosePrice", "CloseDate", "PropertyType", "PropertySubType",
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "City", "PostalCode",
    "CountyOrParish", "StateOrProvince", "ListingKey", "ListPrice",
]

NUMERIC_COLUMNS = [
    "ClosePrice", "LivingArea", "BedroomsTotal",
    "BathroomsTotalInteger", "LotSizeSquareFeet",
]


In [ ]:
# Load and combine all training months.
training_frames = []
for file in train_files:
    month = pd.read_csv(file, usecols=lambda col: col in EDA_COLUMNS, low_memory=False)
    month["SourceMonth"] = str(source_month(file))
    training_frames.append(month)

raw = pd.concat(training_frames, ignore_index=True)
print(f"Selected EDA table: {raw.shape[0]:,} rows × {raw.shape[1]} columns")

#15 selected source columns + 1 SourceMonth column = 16 columns

In [ ]:
# Preview the combined training data.
print("First five combined training rows:")
display(raw.head())

In [ ]:
# Keep residential single-family homes.
sfr_mask = (
    raw["PropertyType"].eq("Residential")
    & raw["PropertySubType"].eq("SingleFamilyResidence")
)
sfr = raw.loc[sfr_mask].copy()

display(pd.DataFrame({
    "Rows": [len(raw), len(sfr), len(raw) - len(sfr)],
}, index=["Before filter", "Kept SFR", "Removed"]))


In [ ]:
# Convert columns to the correct data types.
for column in NUMERIC_COLUMNS + ["ListPrice"]:
    sfr[column] = pd.to_numeric(sfr[column], errors="coerce")
sfr["CloseDate"] = pd.to_datetime(sfr["CloseDate"], errors="coerce")

print("close-date range:", sfr["CloseDate"].min().date(), "to", sfr["CloseDate"].max().date())
display(sfr[NUMERIC_COLUMNS].describe().round(2))

#ListPrice: The seller’s asking price near the time of sale
#ClosePrice: The final amount the buyer actually paid.

In [ ]:
# Check missing and nonpositive values. (zero and negative rows)
quality = pd.DataFrame({
    "Missing %": sfr[NUMERIC_COLUMNS].isna().mean().mul(100),
    "Nonpositive rows": (sfr[NUMERIC_COLUMNS] <= 0).sum(),
}).round(2)
display(quality)

In [ ]:
# Count missing, zero, and negative values.
missing_or_zero = sfr[NUMERIC_COLUMNS].isna() | sfr[NUMERIC_COLUMNS].eq(0)

value_check = pd.DataFrame({
    "Missing rows": sfr[NUMERIC_COLUMNS].isna().sum(),
    "Zero rows": sfr[NUMERIC_COLUMNS].eq(0).sum(),
    "Negative rows": sfr[NUMERIC_COLUMNS].lt(0).sum(),
    "Missing or zero rows": missing_or_zero.sum(),
    "Missing or zero %": missing_or_zero.mean().mul(100).round(2),
})

display(value_check)

In [ ]:
# Summarize rows, months, duplicates, cities, and counties.
overview = pd.Series({
    "SFR rows": len(sfr),
    "Source months": sfr["SourceMonth"].nunique(),
    "Duplicate ListingKey": sfr["ListingKey"].duplicated().sum(),
    "Cities": sfr["City"].nunique(),
    "Counties": sfr["CountyOrParish"].nunique(),
})
display(overview.to_frame("Value"))

# Duplicate ListingKey = 252 means 252 rows have a ListingKey that already appeared in an earlier row.
# Doesnt necessarily mean there are 252 different duplicated listings. It means there are 252 extra repeated rows.
# Listingkey - we wouldnt use it as a model feature b/c the number itself doenst have useful relationship with home's value.

In [ ]:
# Compare low, median, and high numerical values.
# p01: 1st percentile—99% of values are greater than this.
# median: 50th percentile—the middle value.
# p99: 99th percentile—99% of values are below this.

percentiles = sfr[NUMERIC_COLUMNS].quantile([0.01, 0.50, 0.99]).T
percentiles.columns = ["p01", "median", "p99"]
display(percentiles.round(2))


In [ ]:
# Explore feature distributions.
# Plot positive values through the 99th percentile without changing the data.

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, column in zip(axes.ravel(), NUMERIC_COLUMNS):
    values = sfr.loc[sfr[column] > 0, column]
    values = values[values <= values.quantile(0.99)]
    ax.hist(values, bins=40, color="#4C78A8", edgecolor="white")
    ax.set(title=column, xlabel=column, ylabel="Homes")

axes.ravel()[-1].axis("off")
fig.suptitle("Training-set SFR distributions (clipped at p99)", y=1.02)
fig.tight_layout()
plt.show()


In [ ]:
# Examine close price over time.
monthly_price = sfr.groupby("SourceMonth")["ClosePrice"].agg(["count", "median"])
display(monthly_price)

monthly_price["median"].plot(figsize=(10, 4), marker="o", color="#4C78A8")
plt.title("Median SFR close price by source month")
plt.xlabel("Source month")
plt.ylabel("Median ClosePrice ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Examine the log-price distribution.
# log1p compresses the long right tail only for this chart.
positive_prices = sfr.loc[sfr["ClosePrice"] > 0, "ClosePrice"]
plt.figure(figsize=(7, 4))
plt.hist(np.log1p(positive_prices), bins=40, color="#4C78A8", edgecolor="white")
plt.title("Distribution of log1p(ClosePrice)")
plt.xlabel("log1p(ClosePrice)")
plt.ylabel("Homes")
plt.tight_layout()
plt.show()

# Most homes are clustered between log values of approximately 13.5 and 14. ($729,000 - $1,203,000)
# log value 13 → $442,000
# log value 14 → $1,203,000
# log value 15 → $3,269,000

In [ ]:
# Compare living area with close price.
usable = sfr[(sfr["ClosePrice"] > 0) & (sfr["LivingArea"] > 0)]
sample = usable.sample(n=min(10_000, len(usable)), random_state=42)

plt.figure(figsize=(7, 4))
plt.scatter(sample["LivingArea"], sample["ClosePrice"], s=7, alpha=0.2)
plt.title("ClosePrice vs LivingArea (sample)")
plt.xlabel("LivingArea (sq ft)")
plt.ylabel("ClosePrice ($)")
plt.tight_layout()
plt.show()


In [ ]:
# Measure each numerical feature's correlation with close price.
correlations = sfr[NUMERIC_COLUMNS].corr(numeric_only=True)["ClosePrice"]
display(correlations.sort_values(ascending=False).to_frame("Correlation with ClosePrice"))


In [ ]:
# Inspect ListPrice for possible target leakage.
listed = sfr[(sfr["ListPrice"] > 0) & (sfr["ClosePrice"] > 0)].copy()
close_to_list = listed["ClosePrice"] / listed["ListPrice"]

leakage_summary = pd.Series({
    "Rows with both prices": len(listed),
    "Close/List median": close_to_list.median(),
    "Close equals List (%)": listed["ClosePrice"].eq(listed["ListPrice"]).mean() * 100,
    "Price correlation": listed[["ListPrice", "ClosePrice"]].corr().iloc[0, 1],
})
display(leakage_summary.to_frame("Value").round(3))


In [ ]:
# Additional plots
# Examine the spread and outliers in positive close prices.
positive_prices = sfr.loc[sfr["ClosePrice"] > 0, "ClosePrice"]

plt.figure(figsize=(9, 3))
plt.boxplot(positive_prices, vert=False)
plt.xscale("log")
plt.title("Distribution of ClosePrice")
plt.xlabel("ClosePrice ($, logarithmic scale)")
plt.tight_layout()
plt.show()

# Most prices are concentrated around approximately $1 million. 
# However, the chart shows extremely small prices on the left and unusually large prices some approaching $1 billion on the right.

In [ ]:
# Examine where homes are concentrated by living area and close price.
usable = sfr[
    (sfr["LivingArea"] > 0)
    & (sfr["ClosePrice"] > 0)
].copy()

usable = usable[
    (usable["LivingArea"] <= usable["LivingArea"].quantile(0.99))
    & (usable["ClosePrice"] <= usable["ClosePrice"].quantile(0.99))
]

plt.figure(figsize=(8, 5))
plt.hexbin(
    usable["LivingArea"],
    usable["ClosePrice"],
    gridsize=45,
    bins="log",
    cmap="Blues",
    mincnt=1,
)
plt.colorbar(label="Homes per area")
plt.title("ClosePrice versus LivingArea")
plt.xlabel("Living area (sq ft)")
plt.ylabel("ClosePrice ($)")
plt.tight_layout()
plt.show()

# 2 on the y-axix means $2 million.

In [ ]:
# Visualize correlations among the numerical variables.
correlation_matrix = sfr[NUMERIC_COLUMNS].corr()

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(
    correlation_matrix,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)

ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_yticks(range(len(correlation_matrix.columns)))
ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha="right")
ax.set_yticklabels(correlation_matrix.columns)

for row in range(len(correlation_matrix)):
    for column in range(len(correlation_matrix)):
        ax.text(
            column,
            row,
            f"{correlation_matrix.iloc[row, column]:.2f}",
            ha="center",
            va="center",
        )

plt.colorbar(image, label="Correlation")
plt.title("Numerical Feature Correlations")
plt.tight_layout()
plt.show()

## Week 2 findings

- The 26 training files contain **568,074** records before filtering and **285,683** Residential / SingleFamilyResidence records after filtering.
- March 2026 and April 2026 remain untouched for validation and final testing.
- `ClosePrice` is strongly right-skewed, with a median near **$890K** and implausible extremes that require Week 3 cleaning.
- `LivingArea` has the strongest simple relationship with `ClosePrice` among the required numeric fields.
- Missing, nonpositive, extreme, and duplicate values require explicit preprocessing rules.
- `ListPrice` is excluded from model features because it closely tracks the outcome.

Next: `02_data_preprocessing.ipynb` applies the cleaning, imputation, encoding, and scaling rules.